# Isolated Semantic Evidence: Normal vs Wrong Partner

This notebook evaluates the **participant-centric semantic branch** for distinguishing genuine dyadic conversations from **Wrong Partner** pairings using Qwen2.5-Omni-7B.

The controlled evaluation set contains:

- **100 NORMAL cases**, where Participant A is paired with the genuine Participant B from the same conversation;
- **100 WRONG PARTNER cases**, where the same Participant A is paired with a Participant B drawn from a different conversation through a seeded derangement.

Each 120-second participant stream is divided into two aligned 60-second segments.

The notebook retains the original preliminary participant-centric temporal-ensemble pipeline because its cached participant analyses provide the coarse semantic summaries used by the subsequent experiments. That preliminary ensemble is **not part of the final three-way semantic representation comparison reported in the thesis**.

The reported semantic experiments are:

1. **Coarse only** — `speech_content_summary` and `apparent_topic`;
2. **Focused only** — detailed speech content, main and secondary topics, key semantic details, specificity, and uncertainty;
3. **Coarse + focused** — both semantic representations supplied jointly.

Only these three semantic configurations are reported at the end of the notebook. Intermediate diagnostic experiments that were used during development of the focused representation have been removed from this public version.

## Reported results

| Semantic representation | NORMAL correct | WRONG correct | Accuracy |
|---|---:|---:|---:|
| Coarse only | 91/100 | 78/100 | 84.5% |
| Focused only | 47/100 | 100/100 | 73.5% |
| Coarse + focused | 78/100 | 95/100 | **86.5%** |

The combined representation provides the best overall balance: coarse summaries better preserve genuine conversations, whereas focused summaries are substantially more sensitive to concrete semantic mismatches.

## 1. Environment Setup

Install the Qwen2.5-Omni dependencies and verify GPU availability.

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece
!pip install -U qwen-omni-utils decord ffmpeg-python
!apt-get update -qq
!apt-get install -y -qq ffmpeg

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 152.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 136.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 74.3 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
GPU count: 1


## 2. Dataset and Experiment Paths

The experiment uses the processed Seamless Interaction dataset produced by the data-preparation notebook. The dataset is read directly from Google Drive; no additional extraction is required here.

The dedicated `silent/` pool is ignored because this isolated branch evaluates only NORMAL and Wrong Partner cases.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, random, re, shutil, subprocess, time
from tqdm.auto import tqdm

# DATA LOCATION:
# No unzip step is required. The processed data already exist in Google Drive.
DATA_ROOT = Path('/content/drive/MyDrive/seamless_download/data')

# # The silent folder exists inside DATA_ROOT but is not used in this experiment.
# OUT_DIR = Path('/content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_100_100_participant_centric_ensemble')
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# EXPERIMENT PATHS
# ============================================================

EXPERIMENT_NAME = "normal_vs_wrong_balanced_100_100_seed42_v1"

RUN_ROOT = (
    Path("/content/drive/MyDrive/qwen_strategy2")
    / EXPERIMENT_NAME
)

OUT_DIR = RUN_ROOT / "outputs"
PREP_DIR = RUN_ROOT / "preprocessed_segments_60s"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PREP_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_ROOT:", RUN_ROOT)
print("OUT_DIR :", OUT_DIR)
print("PREP_DIR:", PREP_DIR)

assert DATA_ROOT.exists(), f"DATA_ROOT was not found: {DATA_ROOT}"

print("DATA_ROOT:", DATA_ROOT)
print("Top-level folders:", [p.name for p in DATA_ROOT.iterdir() if p.is_dir()][:10])

Mounted at /content/drive
RUN_ROOT: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1
OUT_DIR : /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs
PREP_DIR: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/preprocessed_segments_60s
DATA_ROOT: /content/drive/MyDrive/seamless_download/data
Top-level folders: ['V03_S1166_I00000373', 'V01_S0223_I00000307', 'V01_S0223_I00000127', 'V03_S0180_I00000579', 'V03_S1088_I00000783', 'V03_S1088_I00000579', 'V00_S2048_I00000691', 'V01_S0223_I00000131', 'V01_S0338_I00001106', 'V00_S2049_I00001108']


## 3. Discover Eligible Dyadic Conversations

The processed dataset is scanned for ordinary two-participant conversations and their participant-specific videos.

In [ ]:
def find_first_mp4(folder: Path):
    mp4s = sorted(folder.glob('*.mp4'))
    return mp4s[0] if mp4s else None

def scan_conversations(root: Path, is_silent: bool = False):
    conversations = []
    if not root.exists():
        return conversations

    for conv_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        participants = []
        for part_dir in sorted([p for p in conv_dir.iterdir() if p.is_dir()]):
            mp4 = find_first_mp4(part_dir)
            if mp4 is None:
                continue
            participants.append({
                "conversation_id": conv_dir.name,
                "participant_id": part_dir.name,
                "mp4": str(mp4),
                "is_silent_source": is_silent,
            })
        if participants:
            conversations.append({
                "conversation_id": conv_dir.name,
                "participants": participants,
                "is_silent_source": is_silent,
            })
    return conversations

normal_convs = scan_conversations(DATA_ROOT, is_silent=False)
# IMPORTANT: ignore silent folder completely
normal_convs = [c for c in normal_convs if c["conversation_id"] != "silent"]

print("Normal conversations:", len(normal_convs))
print("Example normal:")
print(json.dumps(normal_convs[0], indent=2) if normal_convs else "NONE")

Normal conversations: 206
Example normal:
{
  "conversation_id": "V00_S2017_I00001160",
  "participants": [
    {
      "conversation_id": "V00_S2017_I00001160",
      "participant_id": "P1273A",
      "mp4": "/content/drive/MyDrive/seamless_download/data/V00_S2017_I00001160/P1273A/V00_S2017_I00001160_P1273A.mp4",
      "is_silent_source": false
    },
    {
      "conversation_id": "V00_S2017_I00001160",
      "participant_id": "P2072A",
      "mp4": "/content/drive/MyDrive/seamless_download/data/V00_S2017_I00001160/P2072A/V00_S2017_I00001160_P2072A.mp4",
      "is_silent_source": false
    }
  ],
  "is_silent_source": false
}


## 4. Construct the Balanced 200-Case Evaluation Set

A matched design is used. For each selected source conversation, Participant A appears once in a genuine NORMAL pair and once in a Wrong Partner pair.

Wrong Partner cases are created by a reproducible seeded derangement of Participant-B streams, ensuring that no participant is paired with their original conversation partner in the anomalous set.

In [ ]:
def safe_name(s: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', s)


# ============================================================
# BALANCED NORMAL / WRONG-PARTNER CASE CONSTRUCTION
# ============================================================

def build_balanced_normal_wrong_cases(convs, n=100, seed=42):
    """
    For each of the n source conversations:

    NORMAL:
        A_i + the genuine B_i

    WRONG_PARTNER:
        the same A_i + B_j from a different conversation

    Each A participant is used:
        - once in NORMAL
        - once in WRONG_PARTNER

    Each B participant is used:
        - once as the genuine partner
        - once as a wrong partner
    """

    eligible = [
        conv for conv in convs
        if len(conv["participants"]) >= 2
    ]

    if len(eligible) < n:
        raise ValueError(
            f"At least {n} conversations with two participants are required, "
            f"but only {len(eligible)} were found."
        )

    # Keep the same first 100 eligible conversations used in the original experiment.
    selected = eligible[:n]

    # Create a reproducible derangement:
    # a permutation with no i -> i assignment.
    rng = random.Random(seed)

    wrong_B_indices = list(range(n))

    while True:
        rng.shuffle(wrong_B_indices)

        if all(i != wrong_B_indices[i] for i in range(n)):
            break

    normal_cases = []
    wrong_cases = []

    for i, conv in enumerate(selected):

        participant_A = conv["participants"][0]
        real_participant_B = conv["participants"][1]

        wrong_conv_idx = wrong_B_indices[i]
        wrong_conv = selected[wrong_conv_idx]
        wrong_participant_B = wrong_conv["participants"][1]

        # -------------------------------
        # NORMAL: A_i + genuine B_i
        # -------------------------------
        normal_cases.append({
            "case_id": f"normal_{i:03d}",
            "pair_index": i,
            "label": "NORMAL",
            "anomaly_type": "none",
            "source": "same_conversation",

            "participant_A": participant_A,
            "participant_B": real_participant_B,

            "A_source_conversation": participant_A["conversation_id"],
            "B_source_conversation": real_participant_B["conversation_id"],
        })

        # -----------------------------------
        # WRONG: the same A_i + unrelated B_j
        # -----------------------------------
        wrong_cases.append({
            "case_id": f"wrong_partner_{i:03d}",
            "pair_index": i,
            "label": "ANOMALOUS",
            "anomaly_type": "wrong_partner",
            "source": "different_conversations",

            "participant_A": participant_A,
            "participant_B": wrong_participant_B,

            "A_source_conversation": participant_A["conversation_id"],
            "B_source_conversation": wrong_participant_B["conversation_id"],
        })

    cases = normal_cases + wrong_cases

    return cases


cases = build_balanced_normal_wrong_cases(
    normal_convs,
    n=100,
    seed=42,
)


# ============================================================
# VALIDATION CHECKS
# ============================================================

normal_subset = [
    case for case in cases
    if case["anomaly_type"] == "none"
]

wrong_subset = [
    case for case in cases
    if case["anomaly_type"] == "wrong_partner"
]


# 100 NORMAL + 100 WRONG
assert len(normal_subset) == 100
assert len(wrong_subset) == 100


# Every NORMAL case comes from the same source conversation
assert all(
    case["participant_A"]["conversation_id"]
    ==
    case["participant_B"]["conversation_id"]
    for case in normal_subset
)


# Every WRONG case combines participants from different source conversations
assert all(
    case["participant_A"]["conversation_id"]
    !=
    case["participant_B"]["conversation_id"]
    for case in wrong_subset
)


# The A participants are identical and aligned across NORMAL and WRONG cases
normal_A = [
    (
        case["participant_A"]["conversation_id"],
        case["participant_A"]["participant_id"],
    )
    for case in normal_subset
]

wrong_A = [
    (
        case["participant_A"]["conversation_id"],
        case["participant_A"]["participant_id"],
    )
    for case in wrong_subset
]

assert normal_A == wrong_A


# There are 100 distinct A participants in the WRONG cases
assert len(set(wrong_A)) == 100


# There are 100 distinct B participants in the WRONG cases
wrong_B = [
    (
        case["participant_B"]["conversation_id"],
        case["participant_B"]["participant_id"],
    )
    for case in wrong_subset
]

assert len(set(wrong_B)) == 100


print("Total cases:", len(cases))
print("NORMAL cases:", len(normal_subset))
print("WRONG_PARTNER cases:", len(wrong_subset))

print(
    "Unique A participants in WRONG cases:",
    len(set(wrong_A))
)

print(
    "Unique B participants in WRONG cases:",
    len(set(wrong_B))
)

print("\nExample paired cases:")

for i in range(3):
    normal_case = normal_subset[i]
    wrong_case = wrong_subset[i]

    print(f"\nConversation index {i}")

    print(
        "NORMAL:",
        normal_case["participant_A"]["conversation_id"],
        normal_case["participant_A"]["participant_id"],
        "+",
        normal_case["participant_B"]["conversation_id"],
        normal_case["participant_B"]["participant_id"],
    )

    print(
        "WRONG :",
        wrong_case["participant_A"]["conversation_id"],
        wrong_case["participant_A"]["participant_id"],
        "+",
        wrong_case["participant_B"]["conversation_id"],
        wrong_case["participant_B"]["participant_id"],
    )


cases_path = (
    OUT_DIR /
    "cases_strategy2_normal_wrong_100_100_balanced.json"
)

cases_path.write_text(
    json.dumps(
        cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nSaved:", cases_path)

Total cases: 200
NORMAL cases: 100
WRONG_PARTNER cases: 100
Unique A participants in WRONG cases: 100
Unique B participants in WRONG cases: 100

Example paired cases:

Conversation index 0
NORMAL: V00_S2017_I00001160 P1273A + V00_S2017_I00001160 P2072A
WRONG : V00_S2017_I00001160 P1273A + V00_S2050_I00001124 P1308A

Conversation index 1
NORMAL: V00_S2017_I00001161 P1273A + V00_S2017_I00001161 P2072A
WRONG : V00_S2017_I00001161 P1273A + V00_S2051_I00001007 P1309A

Conversation index 2
NORMAL: V00_S2017_I00001162 P1273A + V00_S2017_I00001162 P2072A
WRONG : V00_S2017_I00001162 P1273A + V00_S2047_I00001052 P1305A

Saved: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/cases_strategy2_normal_wrong_100_100_balanced.json


In [ ]:
needed = {}
for case in cases:
    for side in ["participant_A", "participant_B"]:
        item = case[side]
        key = (item["conversation_id"], item["participant_id"], item["mp4"])
        needed[key] = item

print("Unique videos needed:", len(needed))

Unique videos needed: 200


## 5. Load Qwen2.5-Omni Thinker

Qwen2.5-Omni is loaded for participant-level audiovisual analysis and subsequent text-only comparison. Speech generation is not used.

In [ ]:
import torch
from transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

print("Loaded:", MODEL_ID)

## 6. Shared Inference Utilities

These helper functions parse structured JSON outputs and provide the audiovisual and text-only Qwen inference calls used throughout the notebook.

In [ ]:
def extract_json_from_text(text: str):
    text = text.strip()

    # remove markdown fences
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"```$", "", text.strip())

    try:
        return json.loads(text)
    except Exception:
        pass

    # find first balanced JSON object
    start = text.find("{")
    if start == -1:
        return {"parse_error": True, "raw_output": text}

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i+1]
                try:
                    return json.loads(candidate)
                except Exception:
                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }

    return {"parse_error": True, "raw_output": text}

def qwen_video_text(video_path: str, prompt: str, max_new_tokens: int = 350):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are Qwen, a virtual human developed by the Qwen Team, "
                        "Alibaba Group, capable of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    )
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "video": video_path, "fps": 7.0},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True
    )

    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    )

    inputs = inputs.to(model.device)

    print("Input tokens:", inputs["input_ids"].shape[1])

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated

def qwen_text_only(prompt: str, max_new_tokens: int = 700):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={
            "padding": True,
        },
    )

    inputs = {
        k: v.to(model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated


## 7. Preliminary Participant-Centric 2×60 s Ensemble

Before the final semantic-only comparisons, each participant is analysed independently over two 60-second segments.

The participant-level prompt generates a compact structured description containing speech content, apparent topic, and interaction-related fields. Corresponding Participant-A and Participant-B segment summaries are then compared.

This preliminary ensemble is retained because its cached participant analyses are the source of the **coarse semantic summaries** used below. Its classification performance is not part of the final three-representation semantic table.

In [ ]:
PARTICIPANT_ANALYSIS_PROMPT_TEMPLATE = """
You are analyzing ONE participant video from a dyadic conversation.

Use both visible behavior and embedded audio.

Main task:
Extract compact evidence that can later be compared with another participant
to decide whether they belong to the same conversation.

Return ONLY valid JSON. Do not include markdown or text outside JSON.

Use exactly this schema:
{{
  "participant_id": "{participant_id}",
  "speaks": true,
  "speaking_amount": "none / low / moderate / high",
  "speech_content_summary": "brief summary of what the participant seems to talk about; use unclear if not understandable",
  "apparent_topic": "short topic label or unclear",
  "asks_questions": true,
  "answers_or_responds": true,
  "visual_engagement": "low / medium / high / unclear",
  "listening_or_reacting": true,
  "waiting_for_other_person": true,
  "interaction_style": "active / passive / mostly listening / unclear",
  "short_description": "one short sentence",
  "confidence": 0.0
}}
"""


# ===========================================
# ===========================================
# ===========================================
# PARTICIPANT_ANALYSIS_PROMPT_TEMPLATE = """
# You are analyzing ONE participant video from a dyadic conversation.

# Use BOTH:
# 1. the embedded audio track
# 2. the visible behavior

# Important distinction:
# - audible_speech_present means spoken words are actually audible in the audio track.
# - visually_appears_to_speak means the participant visually appears to talk, based on mouth movement, gestures, or facial behavior.
# - Do NOT infer audible speech from lip movement.

# Return ONLY valid JSON. Do not include markdown or text outside JSON.

# Use exactly this schema:
# {{
#   "participant_id": "{participant_id}",
#   "audible_speech_present": true,
#   "audible_speech_amount": "none / low / moderate / high",
#   "visually_appears_to_speak": true,
#   "silent_video_with_visible_speaking": true,
#   "speech_content_summary": "brief summary of audible speech content; use none if no audible speech; use unclear if speech is audible but not understandable",
#   "apparent_topic": "short topic label based on audible speech, or none/unclear",
#   "asks_questions": true,
#   "answers_or_responds": true,
#   "visual_engagement": "low / medium / high / unclear",
#   "listening_or_reacting": true,
#   "waiting_for_other_person": true,
#   "interaction_style": "active / passive / mostly listening / unclear",
#   "short_description": "one short sentence",
#   "confidence": 0.0
# }}
# """



# PARTICIPANT_ANALYSIS_PROMPT_TEMPLATE = """
# You are analyzing ONE participant video from a dyadic conversation.

# Use both visible behavior and embedded audio.

# Main task:
# Extract compact evidence that can later be compared with another participant
# to decide whether they belong to the same conversation.

# Important:
# - "speaks" should indicate whether audible speech is actually present.
# - Do NOT mark speaks=true based only on mouth movement, facial expressions,
#   gestures, or apparent speaking behavior.
# - If the participant appears to talk visually but no speech is audible,
#   set speaks=false.
# - Prefer the audio track over visual speaking cues when deciding speaks.
# - speech_content_summary and apparent_topic should be based on audible speech.

# Return ONLY valid JSON. Do not include markdown or text outside JSON.

# Use exactly this schema:
# {{
#   "participant_id": "{participant_id}",
#   "speaks": true,
#   "speaking_amount": "none / low / moderate / high",
#   "speech_content_summary": "brief summary of what the participant seems to talk about; use unclear if not understandable",
#   "apparent_topic": "short topic label or unclear",
#   "asks_questions": true,
#   "answers_or_responds": true,
#   "visual_engagement": "low / medium / high / unclear",
#   "listening_or_reacting": true,
#   "waiting_for_other_person": true,
#   "interaction_style": "active / passive / mostly listening / unclear",
#   "short_description": "one short sentence",
#   "confidence": 0.0
# }}
# """

In [ ]:
COMPARE_PROMPT_TEMPLATE = """
You are evaluating whether two participant summaries belong to the SAME real conversation.

You are given two independent participant analyses.

Important:

NORMAL:
- The participants appear compatible.
- Their topics are compatible.
- One participant may ask questions while the other appears to answer.
- Their speaking styles and interaction patterns appear complementary.
- Absence of strong mismatch evidence should favor NORMAL.

WRONG_PARTNER:
- The participants appear to discuss different topics.
- The participants appear engaged but not in the same interaction.
- Their interaction patterns do not appear complementary.
- Strong semantic mismatch is evidence for WRONG_PARTNER.

You should use:
- speaking_amount
- speech_content_summary
- apparent_topic
- answers_or_responds
- listening_or_reacting
- interaction_style

Do not require exact topic matches.
Do not over-penalize uncertainty.

If evidence is weak or ambiguous, prefer NORMAL.

Return ONLY valid JSON.

Use exactly this schema:
{{
  "label": "NORMAL or ANOMALOUS",
  "anomaly_type": "none / wrong_partner",
  "confidence": 0.0,
  "reasoning": "short explanation",
  "evidence_A": "important evidence",
  "evidence_B": "important evidence"
}}

Participant A:
{analysis_A}

Participant B:
{analysis_B}
"""




# ================================================
# ================================================
# COMPARE_PROMPT_TEMPLATE = """
# You are evaluating whether two participant summaries belong to the SAME real conversation.

# You are given two independent participant analyses.

# Important distinction:
# - audible_speech_present means speech is actually audible in the audio track.
# - visually_appears_to_speak means the participant only appears to talk visually.
# - If someone visually appears to speak but audible_speech_present=false, this is strong evidence for silent_partner.

# NORMAL:
# - The participants appear compatible.
# - Their audible topics are compatible or unclear.
# - One participant may ask questions while the other appears to answer.
# - Their speaking/listening patterns appear complementary.
# - Absence of strong mismatch evidence should favor NORMAL.

# WRONG_PARTNER:
# - Both participants have audible speech, but appear to discuss different topics.
# - Both appear engaged but not in the same interaction.
# - Strong semantic mismatch is evidence for wrong_partner.

# SILENT_PARTNER:
# - One participant has no audible speech while visually appearing to participate or speak.
# - One participant has audible_speech_present=false across the observed segment.
# - Do not treat visual speaking alone as real speech.

# Use:
# - audible_speech_present
# - audible_speech_amount
# - visually_appears_to_speak
# - silent_video_with_visible_speaking
# - speech_content_summary
# - apparent_topic
# - answers_or_responds
# - listening_or_reacting
# - interaction_style

# If evidence is weak or ambiguous, prefer NORMAL.
# But if one participant has no audible speech, prefer silent_partner.

# Return ONLY valid JSON.

# Use exactly this schema:
# {{
#   "label": "NORMAL or ANOMALOUS",
#   "anomaly_type": "none / wrong_partner / silent_partner",
#   "confidence": 0.0,
#   "reasoning": "short explanation",
#   "evidence_A": "important evidence",
#   "evidence_B": "important evidence"
# }}

# Participant A:
# {analysis_A}

# Participant B:
# {analysis_B}
# """





# COMPARE_PROMPT_TEMPLATE = """
# You are evaluating whether two participant summaries belong to the SAME real conversation.

# You are given two independent participant analyses.

# Important:

# NORMAL:
# - The participants appear compatible.
# - Their topics are compatible.
# - One participant may ask questions while the other appears to answer.
# - Their speaking styles and interaction patterns appear complementary.
# - Absence of strong mismatch evidence should favor NORMAL.

# WRONG_PARTNER:
# - The participants appear to discuss different topics.
# - The participants appear engaged but not in the same interaction.
# - Their interaction patterns do not appear complementary.
# - Strong semantic mismatch is evidence for wrong_partner.

# SILENT_PARTNER:
# - One has speak = True and the other one has speak = False.

# You should use:
# - speaking_amount
# - speech_content_summary
# - apparent_topic
# - answers_or_responds
# - listening_or_reacting
# - interaction_style

# Do not require exact topic matches.
# Do not over-penalize uncertainty.

# If evidence is weak or ambiguous, prefer NORMAL.

# Return ONLY valid JSON.

# Use exactly this schema:
# {{
#   "label": "NORMAL or ANOMALOUS",
#   "anomaly_type": "none / wrong_partner / silent_partner",
#   "confidence": 0.0,
#   "reasoning": "short explanation",
#   "evidence_A": "important evidence",
#   "evidence_B": "important evidence"
# }}

# Participant A:
# {analysis_A}

# Participant B:
# {analysis_B}
# """


In [ ]:
# ENSEMBLE_SEGMENTS = [
#     (0, 60),
#     (60, 60)
# ]

# ENSEMBLE_PREP_DIR = Path('/content/drive/MyDrive/qwen_strategy2/preprocessed_segments_60s_normal_wrong_100_100')
# ENSEMBLE_OUT_DIR = OUT_DIR

# # Clean full experiment outputs for this run
# shutil.rmtree(ENSEMBLE_PREP_DIR, ignore_errors=True)
# # Do NOT delete OUT_DIR because cases json is already saved there.
# ENSEMBLE_PREP_DIR.mkdir(parents=True, exist_ok=True)
# ENSEMBLE_OUT_DIR.mkdir(parents=True, exist_ok=True)

# print("Ensemble preprocessing reset.")
# print("ENSEMBLE_PREP_DIR:", ENSEMBLE_PREP_DIR)
# print("ENSEMBLE_OUT_DIR :", ENSEMBLE_OUT_DIR)


ENSEMBLE_SEGMENTS = [
    (0, 60),
    (60, 60),
]

ENSEMBLE_PREP_DIR = PREP_DIR
ENSEMBLE_OUT_DIR = OUT_DIR

ENSEMBLE_PREP_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Ensemble directories ready.")
print("ENSEMBLE_PREP_DIR:", ENSEMBLE_PREP_DIR)
print("ENSEMBLE_OUT_DIR :", ENSEMBLE_OUT_DIR)

Ensemble directories ready.
ENSEMBLE_PREP_DIR: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/preprocessed_segments_60s
ENSEMBLE_OUT_DIR : /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs


In [ ]:
def segmented_path(item, segment_idx):
    conv = safe_name(item["conversation_id"])
    part = safe_name(item["participant_id"])
    out_dir = ENSEMBLE_PREP_DIR / conv
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f"{conv}_{part}_seg{segment_idx}_60s.mp4"

def video_has_audio(path):
    cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "a",
        "-show_entries", "stream=index",
        "-of", "csv=p=0",
        str(path)
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return bool(result.stdout.strip())


def preprocess_video_segment(input_path: str, output_path: Path, start: int, seconds: int = 60, fps: int = 30):
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and output_path.stat().st_size > 0:
        if video_has_audio(output_path):
            return str(output_path)
        else:
            output_path.unlink(missing_ok=True)

    cmd = [
        "ffmpeg", "-y",
        "-ss", str(start),
        "-i", input_path,
        "-t", str(seconds),
        #"-vf", f"scale=720:-2,fps={fps},format=yuv420p",
        "-vf", f"scale=360:-2,fps={fps},format=yuv420p",
        "-c:v", "libx264", "-crf", "23", "-preset", "veryfast",
        "-c:a", "aac", "-ar", "16000", "-ac", "1",
        str(output_path)
    ]

    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    if result.returncode != 0 or not output_path.exists() or output_path.stat().st_size == 0:
        print("Empty or failed segment:", input_path, "start:", start)
        return None

    if not video_has_audio(output_path):
        print("Segment has no audio, skipping:", output_path)
        output_path.unlink(missing_ok=True)
        return None

    return str(output_path)

In [ ]:
for item in tqdm(list(needed.values())):
    item["segments"] = []

    for seg_idx, (start, duration) in enumerate(ENSEMBLE_SEGMENTS):
        out = segmented_path(item, seg_idx)
        seg_path = preprocess_video_segment(
            input_path=item["mp4"],
            output_path=out,
            start=start,
            seconds=duration,
            fps=30,
        )

        if seg_path is not None:
            item["segments"].append({
                "segment_idx": seg_idx,
                "start": start,
                "duration": duration,
                "path": seg_path,
            })

print("Segment preprocessing done.")

Segment preprocessing done.


In [ ]:
ensemble_participant_cache_path = ENSEMBLE_OUT_DIR / "participant_segment_analyses.json"

if ensemble_participant_cache_path.exists():
    ensemble_participant_cache = json.loads(
        ensemble_participant_cache_path.read_text(encoding="utf-8")
    )
else:
    ensemble_participant_cache = {}

def segment_key(item, seg):
    return f'{item["conversation_id"]}::{item["participant_id"]}::seg{seg["segment_idx"]}::{seg["path"]}'

def analyze_participant_segment(item, seg):
    key = segment_key(item, seg)

    if key in ensemble_participant_cache:
        return ensemble_participant_cache[key]

    prompt = PARTICIPANT_ANALYSIS_PROMPT_TEMPLATE.format(
        participant_id=f'{item["participant_id"]}_seg{seg["segment_idx"]}'
    )

    raw = qwen_video_text(
        video_path=seg["path"],
        prompt=prompt,
        max_new_tokens=300,
    )

    parsed = extract_json_from_text(raw)

    record = {
        "conversation_id": item["conversation_id"],
        "participant_id": item["participant_id"],
        "segment_idx": seg["segment_idx"],
        "start": seg["start"],
        "duration": seg["duration"],
        "video": seg["path"],
        "raw_output": raw,
        "parsed": parsed,
    }

    ensemble_participant_cache[key] = record
    ensemble_participant_cache_path.write_text(
        json.dumps(ensemble_participant_cache, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    return record

# # Run participant analysis for all segments first

for item in tqdm(list(needed.values())):
    for seg in item["segments"]:
        _ = analyze_participant_segment(item, seg)

print("Total segment analyses:", len(ensemble_participant_cache))

Total segment analyses: 400


In [ ]:
ensemble_results_path = ENSEMBLE_OUT_DIR / "strategy2_segment_pair_results.json"

if ensemble_results_path.exists():
    ensemble_results = json.loads(
        ensemble_results_path.read_text(encoding="utf-8")
    )
else:
    ensemble_results = []

done_segment_cases = {
    (r["case_id"], r["segment_idx"]) for r in ensemble_results
}



def compare_case_segment(case, segment_idx):
    if (case["case_id"], segment_idx) in done_segment_cases:
        return next(
            r for r in ensemble_results
            if r["case_id"] == case["case_id"] and r["segment_idx"] == segment_idx
        )

    item_A = case["participant_A"]
    item_B = case["participant_B"]

    if segment_idx >= len(item_A.get("segments", [])):
        return None
    if segment_idx >= len(item_B.get("segments", [])):
        return None

    seg_A = item_A["segments"][segment_idx]
    seg_B = item_B["segments"][segment_idx]

    rec_A = analyze_participant_segment(item_A, seg_A)
    rec_B = analyze_participant_segment(item_B, seg_B)

    analysis_A = json.dumps(rec_A["parsed"], ensure_ascii=False, indent=2)
    analysis_B = json.dumps(rec_B["parsed"], ensure_ascii=False, indent=2)

    prompt = COMPARE_PROMPT_TEMPLATE.format(
        analysis_A=analysis_A,
        analysis_B=analysis_B,
    )

    raw = qwen_text_only(prompt, max_new_tokens=300)
    parsed = extract_json_from_text(raw)

    result = {
        "case_id": case["case_id"],
        "segment_idx": segment_idx,
        "gold_label": case["label"],
        "gold_anomaly_type": case["anomaly_type"],
        "participant_A": case["participant_A"]["participant_id"],
        "participant_B": case["participant_B"]["participant_id"],
        "raw_output": raw,
        "parsed": parsed,
    }

    ensemble_results.append(result)
    ensemble_results_path.write_text(
        json.dumps(ensemble_results, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    done_segment_cases.add((case["case_id"], segment_idx))
    return result

for case in tqdm(cases):
    for seg_idx in range(len(ENSEMBLE_SEGMENTS)):
        _ = compare_case_segment(case, seg_idx)

print("Saved:", ensemble_results_path)
print("Segment-level comparison results:", len(ensemble_results))

Saved: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/strategy2_segment_pair_results.json
Segment-level comparison results: 400


# Experiment 1 — Coarse-Only Semantic Evidence

The first reported semantic experiment removes the segment-level voting stage and evaluates the complete 120-second interaction jointly.

For each participant and each 60-second segment, the classifier receives only:

- `speech_content_summary`
- `apparent_topic`

No video, visual behaviour, interaction style, speaking amount, or other participant fields are provided to the final semantic classifier.

The two synchronized segments are interpreted jointly rather than by majority vote.

In [ ]:
# ============================================================
# NEW EXPERIMENT:
# 120-SECOND SEMANTIC-ONLY WRONG-PARTNER CLASSIFICATION
#
# Input:
#   ONLY speech_content_summary + apparent_topic
#   from both 60-second segments of both participants.
#
# No video inference.
# No new participant summaries.
# No segment-level voting.
# ============================================================

SEMANTIC_120S_EXPERIMENT_VERSION = (
    "semantic_only_four_summaries_120s_v1"
)

SEMANTIC_120S_RESULTS_PATH = (
    ENSEMBLE_OUT_DIR
    / "semantic_only_four_summaries_120s_results_v1.json"
)

SEMANTIC_120S_CSV_PATH = (
    ENSEMBLE_OUT_DIR
    / "semantic_only_four_summaries_120s_results_v1.csv"
)


SEMANTIC_120S_PROMPT_TEMPLATE = """
You are determining whether two people belong to the SAME real
120-second dyadic conversation.

You are given semantic summaries from two synchronized 60-second
segments for Participant A and Participant B.

You must use ONLY:
- speech_content_summary
- apparent_topic

Do not use visual engagement, interaction style, speaking amount,
listening behavior, or any other information.

The summaries were independently generated and may sometimes be broad,
imperfect, or uncertain.

IMPORTANT INTERPRETATION

NORMAL:
- The two participants' content can plausibly belong to the same conversation.
- They may discuss different aspects of a shared subject.
- One participant's content may plausibly respond to, elaborate on, or provide
  context for the other participant's content.
- A conversation may naturally change topic between Segment 0 and Segment 1.
- Exact word or topic-label matching is not required.

WRONG_PARTNER:
- The participants repeatedly discuss unrelated concrete subjects.
- There is no plausible shared conversational context across the 120 seconds.
- Both synchronized segments show semantic mismatch, or one segment shows a
  strong concrete mismatch while the other provides no credible compatibility.
- The fact that both participants discuss personal experiences, preferences,
  opinions, daily life, products, or general topics is NOT by itself evidence
  that they belong to the same conversation.

GENERIC OR WEAK SUMMARIES

Broad labels such as:
- personal experiences
- personal preferences
- general discussion
- daily life
- opinions
- lifestyle
- personal well-being

must not be treated as evidence of compatibility unless the concrete content
also provides a plausible semantic connection.

If one segment is vague or generic, treat that segment as
INSUFFICIENT_EVIDENCE rather than as evidence for NORMAL.

Evaluate the complete 120-second pattern. Do not use a simple vote between
the two segments.

Return ONLY valid JSON with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS",
  "anomaly_type": "none or wrong_partner",
  "confidence": 0.0,
  "segment_0_assessment": "compatible / mismatch / insufficient_evidence",
  "segment_1_assessment": "compatible / mismatch / insufficient_evidence",
  "cross_segment_assessment": "short assessment of the complete 120-second semantic relationship",
  "reasoning": "brief evidence-based explanation"
}}

SEMANTIC INPUT:

{semantic_input}
"""

In [ ]:
# ============================================================
# LOAD CACHED SEMANTIC SUMMARIES
# ============================================================

def get_cached_semantic_fields(
    participant_item,
    segment_idx,
):
    """
    Retrieves ONLY:
      - speech_content_summary
      - apparent_topic

    from the already completed participant-summary cache.

    This function does NOT call Qwen.
    """

    conversation_id = participant_item["conversation_id"]
    participant_id = participant_item["participant_id"]

    matching_records = []

    for record in ensemble_participant_cache.values():
        if not isinstance(record, dict):
            continue

        if (
            record.get("conversation_id") == conversation_id
            and record.get("participant_id") == participant_id
            and int(record.get("segment_idx", -1)) == segment_idx
        ):
            matching_records.append(record)

    if not matching_records:
        raise KeyError(
            "No cached participant summary found for:\n"
            f"conversation={conversation_id}\n"
            f"participant={participant_id}\n"
            f"segment_idx={segment_idx}"
        )

    # In case old and new cache entries coexist, use the latest valid one.
    valid_records = [
        record
        for record in matching_records
        if isinstance(record.get("parsed"), dict)
    ]

    if not valid_records:
        raise RuntimeError(
            "Cached records exist, but none contains valid parsed JSON for:\n"
            f"conversation={conversation_id}\n"
            f"participant={participant_id}\n"
            f"segment_idx={segment_idx}"
        )

    record = valid_records[-1]
    parsed = record["parsed"]

    speech_summary = parsed.get(
        "speech_content_summary",
        "unclear",
    )

    apparent_topic = parsed.get(
        "apparent_topic",
        "unclear",
    )

    if speech_summary is None:
        speech_summary = "unclear"

    if apparent_topic is None:
        apparent_topic = "unclear"

    return {
        "speech_content_summary": str(speech_summary).strip(),
        "apparent_topic": str(apparent_topic).strip(),
    }


def build_case_semantic_input(case):
    """
    Creates the exact input given to the 120-second semantic classifier.
    """

    semantic_input = {
        "participant_A": {
            "segment_0_0_to_60_seconds": (
                get_cached_semantic_fields(
                    case["participant_A"],
                    segment_idx=0,
                )
            ),
            "segment_1_60_to_120_seconds": (
                get_cached_semantic_fields(
                    case["participant_A"],
                    segment_idx=1,
                )
            ),
        },

        "participant_B": {
            "segment_0_0_to_60_seconds": (
                get_cached_semantic_fields(
                    case["participant_B"],
                    segment_idx=0,
                )
            ),
            "segment_1_60_to_120_seconds": (
                get_cached_semantic_fields(
                    case["participant_B"],
                    segment_idx=1,
                )
            ),
        },
    }

    return semantic_input


# Sanity check on one case
example_semantic_input = build_case_semantic_input(
    cases[0]
)

print(json.dumps(
    example_semantic_input,
    indent=2,
    ensure_ascii=False,
))

In [ ]:
# ============================================================
# RUN 120-SECOND SEMANTIC-ONLY CLASSIFICATION
# ============================================================

if SEMANTIC_120S_RESULTS_PATH.exists():
    semantic_120s_results = json.loads(
        SEMANTIC_120S_RESULTS_PATH.read_text(
            encoding="utf-8"
        )
    )
else:
    semantic_120s_results = []


# Keep only results from this exact prompt/experiment version.
compatible_existing_results = {
    result["case_id"]: result
    for result in semantic_120s_results
    if (
        result.get("experiment_version")
        == SEMANTIC_120S_EXPERIMENT_VERSION
    )
}

print(
    "Existing compatible results:",
    len(compatible_existing_results),
)


# Fixed shuffled inference order.
# This does not change the dataset or pairing.
semantic_cases_to_run = list(cases)

semantic_rng = random.Random(42 + 5000)
semantic_rng.shuffle(semantic_cases_to_run)


for case in tqdm(
    semantic_cases_to_run,
    desc="120s semantic-only classification",
):
    case_id = case["case_id"]

    if case_id in compatible_existing_results:
        continue

    semantic_input = build_case_semantic_input(case)

    prompt = SEMANTIC_120S_PROMPT_TEMPLATE.format(
        semantic_input=json.dumps(
            semantic_input,
            indent=2,
            ensure_ascii=False,
        )
    )

    raw_output = qwen_text_only(
        prompt,
        max_new_tokens=350,
    )

    parsed_output = extract_json_from_text(
        raw_output
    )

    result = {
        "experiment_version": (
            SEMANTIC_120S_EXPERIMENT_VERSION
        ),

        "case_id": case_id,

        # Gold values are saved only for later evaluation.
        # They were not included in the prompt.
        "gold_label": case["label"],
        "gold_anomaly_type": case["anomaly_type"],

        "participant_A_conversation": (
            case["participant_A"]["conversation_id"]
        ),
        "participant_A_id": (
            case["participant_A"]["participant_id"]
        ),

        "participant_B_conversation": (
            case["participant_B"]["conversation_id"]
        ),
        "participant_B_id": (
            case["participant_B"]["participant_id"]
        ),

        "semantic_input": semantic_input,
        "raw_output": raw_output,
        "parsed": parsed_output,
    }

    semantic_120s_results.append(result)
    compatible_existing_results[case_id] = result

    # Incremental save so the experiment can resume.
    SEMANTIC_120S_RESULTS_PATH.write_text(
        json.dumps(
            semantic_120s_results,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


print(
    "Completed compatible results:",
    len(compatible_existing_results),
)

print(
    "Saved:",
    SEMANTIC_120S_RESULTS_PATH,
)

Existing compatible results: 200


Completed compatible results: 200
Saved: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/semantic_only_four_summaries_120s_results_v1.json


In [ ]:
# ============================================================
# EVALUATE 120-SECOND SEMANTIC-ONLY EXPERIMENT
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)


def normalize_semantic_120s_prediction(parsed):
    if not isinstance(parsed, dict):
        return None

    label = str(
        parsed.get("label", "")
    ).strip().upper()

    anomaly_type = str(
        parsed.get("anomaly_type", "")
    ).strip().lower()

    normal_aliases = {
        "NORMAL",
        "NONE",
    }

    anomalous_aliases = {
        "ANOMALOUS",
        "ANOMALY",
        "WRONG_PARTNER",
        "WRONG PARTNER",
    }

    if label in normal_aliases:
        return "NORMAL"

    if label in anomalous_aliases:
        return "ANOMALOUS"

    if anomaly_type in {
        "wrong_partner",
        "wrong partner",
    }:
        return "ANOMALOUS"

    if anomaly_type in {
        "none",
        "normal",
    }:
        return "NORMAL"

    return None


semantic_120s_rows = []

for case_id, result in compatible_existing_results.items():
    parsed = result.get("parsed", {})

    semantic_120s_rows.append({
        "case_id": case_id,
        "gold_label": result["gold_label"],
        "gold_anomaly_type": (
            result["gold_anomaly_type"]
        ),

        "predicted_label": (
            normalize_semantic_120s_prediction(
                parsed
            )
        ),

        "predicted_anomaly_type": (
            parsed.get("anomaly_type")
            if isinstance(parsed, dict)
            else None
        ),

        "confidence": (
            parsed.get("confidence")
            if isinstance(parsed, dict)
            else None
        ),

        "segment_0_assessment": (
            parsed.get("segment_0_assessment")
            if isinstance(parsed, dict)
            else None
        ),

        "segment_1_assessment": (
            parsed.get("segment_1_assessment")
            if isinstance(parsed, dict)
            else None
        ),

        "cross_segment_assessment": (
            parsed.get("cross_segment_assessment")
            if isinstance(parsed, dict)
            else None
        ),

        "reasoning": (
            parsed.get("reasoning")
            if isinstance(parsed, dict)
            else None
        ),
    })


semantic_120s_df = (
    pd.DataFrame(semantic_120s_rows)
    .sort_values("case_id")
    .reset_index(drop=True)
)

display(semantic_120s_df)


valid_semantic_120s = semantic_120s_df[
    semantic_120s_df["predicted_label"].isin(
        ["NORMAL", "ANOMALOUS"]
    )
].copy()


print(
    "Total expected cases:",
    len(cases),
)

print(
    "Valid parsed predictions:",
    len(valid_semantic_120s),
)

print(
    "Parse/normalization failures:",
    len(semantic_120s_df)
    - len(valid_semantic_120s),
)


assert len(semantic_120s_df) == len(cases), (
    f"Expected {len(cases)} results, "
    f"found {len(semantic_120s_df)}."
)


print(
    "\nAccuracy:",
    accuracy_score(
        valid_semantic_120s["gold_label"],
        valid_semantic_120s["predicted_label"],
    )
)

print("\nClassification report:")

print(classification_report(
    valid_semantic_120s["gold_label"],
    valid_semantic_120s["predicted_label"],
    labels=["ANOMALOUS", "NORMAL"],
    zero_division=0,
))

print("\nConfusion matrix [NORMAL, ANOMALOUS]:")

semantic_120s_cm = confusion_matrix(
    valid_semantic_120s["gold_label"],
    valid_semantic_120s["predicted_label"],
    labels=["NORMAL", "ANOMALOUS"],
)

print(semantic_120s_cm)


semantic_120s_df.to_csv(
    SEMANTIC_120S_CSV_PATH,
    index=False,
)

print(
    "\nSaved CSV:",
    SEMANTIC_120S_CSV_PATH,
)

Total expected cases: 200
Valid parsed predictions: 200
Parse/normalization failures: 0

Accuracy: 0.845

Classification report:
              precision    recall  f1-score   support

   ANOMALOUS       0.90      0.78      0.83       100
      NORMAL       0.81      0.91      0.85       100

    accuracy                           0.84       200
   macro avg       0.85      0.84      0.84       200
weighted avg       0.85      0.84      0.84       200


Confusion matrix [NORMAL, ANOMALOUS]:
[[91  9]
 [22 78]]

Saved CSV: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/semantic_only_four_summaries_120s_results_v1.csv


### Coarse-only result

The coarse representation correctly preserves **91/100 NORMAL** interactions and detects **78/100 Wrong Partner** cases, for an overall accuracy of **84.5%**.

The result shows that broad speech-content and topic summaries already contain substantial relational information, while their relatively conservative granularity favours preservation of genuine conversations.

# Experiment 2 — Focused-Only Semantic Evidence

The second reported experiment replaces the coarse representation with a more specific participant summary.

For each participant segment, the focused representation captures:

- `detailed_speech_summary`
- `main_topic`
- `secondary_topics`
- `key_semantic_details`
- `summary_specificity`
- `unclear_content`

The goal is to preserve concrete people, events, activities, products, opinions, questions, decisions, and other details that may be compressed away by broader summaries.

Development-only diagnostic experiments used while designing this representation are intentionally omitted from this public notebook. The cells below contain only the reusable focused-summary functions and the final full 200-case focused-only experiment.

In [ ]:
# ============================================================
# FOCUSED SEMANTIC PARTICIPANT SUMMARY PROMPT
# ============================================================

FOCUSED_SUMMARY_VERSION = (
    "focused_participant_semantic_summary_v1"
)

FOCUSED_SUMMARY_CACHE_PATH = (
    ENSEMBLE_OUT_DIR
    / "focused_participant_semantic_summaries_FN22_v1.json"
)


FOCUSED_SEMANTIC_SUMMARY_PROMPT_TEMPLATE = """
You are analyzing ONE 60-second participant video from a dyadic conversation.

Use the embedded audio as the primary source of information.
Use visible behavior only when it helps interpret what the participant is saying.

Your task is to produce an accurate, specific semantic description of what
this participant says during this segment.

This output will later be compared with the semantic description of another
participant to determine whether they plausibly belong to the same conversation.

Therefore:

- Describe the concrete content of the speech, not merely a broad category.
- Preserve important people, objects, events, activities, opinions, decisions,
  questions, problems, examples, places, products, and contextual details.
- Explain what the participant says clearly enough that another model can
  compare it with another participant's speech.
- If the participant discusses multiple subjects, include each important one.
- Distinguish clearly between what is understandable and what is uncertain.
- Do not replace specific content with vague phrases such as:
  "personal experiences",
  "personal preferences",
  "general discussion",
  "daily life",
  "various topics",
  "opinions",
  or "lifestyle".
- Do not infer that two broad personal subjects are the same topic.
- Do not invent details that are not supported by the audio.
- Do not decide whether this is a normal or wrong-partner conversation.
- Analyze only the participant in this video.
- If no audible speech is present, set speaks=false and state that the semantic
  content is unavailable.

Return ONLY valid JSON. Do not include markdown or text outside JSON.

Use exactly this schema:

{{
  "participant_id": "{participant_id}",
  "speaks": true,
  "detailed_speech_summary": "specific and informative description of what the participant says",
  "main_topic": "specific primary subject or unclear",
  "secondary_topics": [
    "specific additional subject"
  ],
  "key_semantic_details": [
    "specific person, object, event, opinion, question, decision, action, place, product, or claim"
  ],
  "summary_specificity": "high / medium / low",
  "unclear_content": "brief description of anything that could not be understood, or none",
  "confidence": 0.0
}}
"""

In [ ]:
# ============================================================
# FOCUSED SUMMARY CACHE AND SEGMENT RESOLUTION
# ============================================================

if FOCUSED_SUMMARY_CACHE_PATH.exists():
    focused_summary_cache = json.loads(
        FOCUSED_SUMMARY_CACHE_PATH.read_text(
            encoding="utf-8"
        )
    )
else:
    focused_summary_cache = {}


def resolve_participant_segment(
    participant_item,
    segment_idx,
):
    """
    Finds the existing 60-second segment.

    If the segment entry/path is missing, it reconstructs only that
    FFmpeg segment using the already defined preprocessing function.
    """

    segment_idx = int(segment_idx)

    if segment_idx not in {0, 1}:
        raise ValueError(
            "segment_idx must be 0 or 1."
        )

    segment = None

    for candidate in participant_item.get(
        "segments",
        [],
    ):
        if int(candidate["segment_idx"]) == segment_idx:
            segment = candidate
            break

    start, duration = ENSEMBLE_SEGMENTS[segment_idx]

    if segment is None:
        segment = {
            "segment_idx": segment_idx,
            "start": start,
            "duration": duration,
            "path": None,
        }

    segment_path = segment.get("path")

    if (
        segment_path is None
        or not Path(segment_path).exists()
        or Path(segment_path).stat().st_size == 0
    ):
        output_path = segmented_path(
            participant_item,
            segment_idx,
        )

        segment_path = preprocess_video_segment(
            input_path=participant_item["mp4"],
            output_path=output_path,
            start=start,
            seconds=duration,
            fps=30,
        )

        if segment_path is None:
            raise RuntimeError(
                "Could not construct segment for:\n"
                f'{participant_item["conversation_id"]} / '
                f'{participant_item["participant_id"]} / '
                f"segment {segment_idx}"
            )

        segment["path"] = segment_path

    return segment


def focused_summary_key(
    participant_item,
    segment,
):
    return (
        f"{FOCUSED_SUMMARY_VERSION}::"
        f'{participant_item["conversation_id"]}::'
        f'{participant_item["participant_id"]}::'
        f'seg{segment["segment_idx"]}::'
        f'{segment["path"]}'
    )


def save_focused_summary_cache():
    FOCUSED_SUMMARY_CACHE_PATH.write_text(
        json.dumps(
            focused_summary_cache,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


print(
    "Existing focused summary records:",
    len(focused_summary_cache),
)

print(
    "Focused summary cache:",
    FOCUSED_SUMMARY_CACHE_PATH,
)

Existing focused summary records: 88
Focused summary cache: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/focused_participant_semantic_summaries_FN22_v1.json


In [ ]:
# ============================================================
# GENERATE ONE FOCUSED SUMMARY FOR ONE PARTICIPANT SEGMENT
# ============================================================

def analyze_focused_participant_segment(
    participant_item,
    segment_idx,
    force_rerun=False,
):
    segment = resolve_participant_segment(
        participant_item,
        segment_idx,
    )

    key = focused_summary_key(
        participant_item,
        segment,
    )

    if (
        not force_rerun
        and key in focused_summary_cache
    ):
        return focused_summary_cache[key]

    prompt = (
        FOCUSED_SEMANTIC_SUMMARY_PROMPT_TEMPLATE
        .format(
            participant_id=(
                f'{participant_item["participant_id"]}'
                f'_seg{segment_idx}'
            )
        )
    )

    raw_output = qwen_video_text(
        video_path=segment["path"],
        prompt=prompt,
        max_new_tokens=500,
    )

    parsed_output = extract_json_from_text(
        raw_output
    )

    record = {
        "experiment_version": (
            FOCUSED_SUMMARY_VERSION
        ),

        "conversation_id": (
            participant_item["conversation_id"]
        ),

        "participant_id": (
            participant_item["participant_id"]
        ),

        "segment_idx": int(segment_idx),
        "start": segment["start"],
        "duration": segment["duration"],
        "video": segment["path"],

        "raw_output": raw_output,
        "parsed": parsed_output,
    }

    focused_summary_cache[key] = record
    save_focused_summary_cache()

    return record

In [ ]:
# ============================================================
# BUILD FOUR-SUMMARY INPUT FOR EACH CASE
# ============================================================

FOCUSED_SEMANTIC_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
]


def get_focused_summary_record(
    participant_item,
    segment_idx,
):
    segment = resolve_participant_segment(
        participant_item,
        segment_idx,
    )

    key = focused_summary_key(
        participant_item,
        segment,
    )

    if key not in focused_summary_cache:
        return analyze_focused_participant_segment(
            participant_item,
            segment_idx,
        )

    return focused_summary_cache[key]


def focused_semantic_projection(record):
    parsed = record.get("parsed", {})

    if (
        not isinstance(parsed, dict)
        or parsed.get("parse_error")
    ):
        return {
            "detailed_speech_summary": "unclear",
            "main_topic": "unclear",
            "secondary_topics": [],
            "key_semantic_details": [],
            "summary_specificity": "low",
            "unclear_content": (
                "The participant summary could not "
                "be parsed reliably."
            ),
        }

    return {
        field: parsed.get(field)
        for field in FOCUSED_SEMANTIC_FIELDS
    }


def build_focused_case_input(case):
    result = {
        "participant_A": {},
        "participant_B": {},
    }

    for participant_role in ["A", "B"]:
        participant_item = case[
            f"participant_{participant_role}"
        ]

        record_0 = get_focused_summary_record(
            participant_item,
            segment_idx=0,
        )

        record_1 = get_focused_summary_record(
            participant_item,
            segment_idx=1,
        )

        result[
            f"participant_{participant_role}"
        ] = {
            "segment_0_0_to_60_seconds": (
                focused_semantic_projection(
                    record_0
                )
            ),

            "segment_1_60_to_120_seconds": (
                focused_semantic_projection(
                    record_1
                )
            ),
        }

    return result

In [ ]:
# ============================================================
# FOCUSED 120-SECOND WRONG-PARTNER CLASSIFIER
# ============================================================

FOCUSED_CLASSIFIER_VERSION = (
    "focused_semantic_four_summaries_FN22_v1"
)

FOCUSED_CLASSIFICATION_RESULTS_PATH = (
    ENSEMBLE_OUT_DIR
    / "focused_semantic_120s_FN22_results_v1.json"
)

FOCUSED_CLASSIFICATION_CSV_PATH = (
    ENSEMBLE_OUT_DIR
    / "focused_semantic_120s_FN22_results_v1.csv"
)


FOCUSED_120S_COMPARISON_PROMPT_TEMPLATE = """
You are determining whether two people plausibly belong to the SAME real
120-second dyadic conversation.

You are given detailed semantic descriptions of what each participant says
during two synchronized 60-second segments.

The descriptions were generated independently from each participant's video.

Your task is to determine whether the participants are:

NORMAL:
- plausibly participating in the same conversation;

or

WRONG_PARTNER:
- taken from different, unrelated conversations.

IMPORTANT REASONING RULES

1. Compare concrete semantic content, not merely broad categories.

2. Shared generic categories such as:
   - personal experiences
   - preferences
   - daily life
   - opinions
   - lifestyle
   - personal well-being
   - work
   are not sufficient evidence that the participants belong to the same
   conversation.

3. Look for concrete shared or complementary context:
   - the same person;
   - the same event;
   - the same product;
   - the same location;
   - the same problem;
   - the same decision;
   - the same activity;
   - a plausible question-and-answer relationship;
   - a plausible continuation or elaboration.

4. Participants in a real conversation do not need to use identical words.
   One participant may answer, clarify, disagree, provide an example, or discuss
   a different aspect of the same subject.

5. If one description is vague or uncertain, treat it as
   INSUFFICIENT_EVIDENCE, not automatically as evidence for NORMAL.

6. Repeated concrete topic mismatches across the synchronized segments are
   strong evidence of WRONG_PARTNER.

7. One strong concrete mismatch combined with an uninformative second segment
   can also support WRONG_PARTNER.

8. Do not use visual engagement, interaction style, speaking amount, or facial
   behavior. Use only the semantic descriptions provided below.

9. Evaluate the complete 120-second pattern. Do not use a mechanical majority
   vote.

Return ONLY valid JSON with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS",
  "anomaly_type": "none or wrong_partner",
  "confidence": 0.0,
  "segment_0_assessment": "compatible / mismatch / insufficient_evidence",
  "segment_1_assessment": "compatible / mismatch / insufficient_evidence",
  "cross_segment_assessment": "brief assessment of the complete 120-second semantic relationship",
  "decisive_evidence": [
    "specific semantic evidence supporting the decision"
  ],
  "reasoning": "brief evidence-based explanation"
}}

SEMANTIC INPUT:

{semantic_input}
"""

In [ ]:
# ============================================================
# NORMALIZE FOCUSED-ONLY PREDICTIONS
# ============================================================

def normalize_focused_prediction(parsed):
    if not isinstance(parsed, dict):
        return None

    label = str(
        parsed.get("label", "")
    ).strip().upper()

    anomaly_type = str(
        parsed.get("anomaly_type", "")
    ).strip().lower()

    if label == "NORMAL":
        return "NORMAL"

    if label in {
        "ANOMALOUS",
        "ANOMALY",
        "WRONG_PARTNER",
        "WRONG PARTNER",
    }:
        return "ANOMALOUS"

    if anomaly_type in {
        "wrong_partner",
        "wrong partner",
    }:
        return "ANOMALOUS"

    if anomaly_type in {
        "none",
        "normal",
    }:
        return "NORMAL"

    return None

## 8. Generate and Validate Focused Summaries for All 200 Cases

The focused participant summaries are generated or loaded from cache for all required participant-segment combinations before conversation-level classification.

In [ ]:
# ============================================================
# FULL 200-CASE FOCUSED SEMANTIC EXPERIMENT
# CONFIGURATION AND CACHE INITIALIZATION
# ============================================================

from pathlib import Path
from tqdm.auto import tqdm

import copy
import json
import random
import pandas as pd


# ------------------------------------------------------------
# Verify that the diagnostic experiment cells were executed
# ------------------------------------------------------------

required_objects = [
    "cases",
    "ENSEMBLE_OUT_DIR",
    "FOCUSED_SUMMARY_VERSION",
    "FOCUSED_SEMANTIC_SUMMARY_PROMPT_TEMPLATE",
    "FOCUSED_120S_COMPARISON_PROMPT_TEMPLATE",
    "resolve_participant_segment",
    "focused_summary_key",
    "analyze_focused_participant_segment",
    "build_focused_case_input",
    "normalize_focused_prediction",
    "qwen_video_text",
    "qwen_text_only",
    "extract_json_from_text",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the previous focused diagnostic cells first. "
        f"Missing objects: {missing_objects}"
    )


assert len(cases) == 200, (
    f"Expected 200 cases, found {len(cases)}."
)

assert sum(
    case["label"] == "NORMAL"
    for case in cases
) == 100

assert sum(
    case["label"] == "ANOMALOUS"
    for case in cases
) == 100


# ------------------------------------------------------------
# New ALL-200 summary cache
# ------------------------------------------------------------

FOCUSED_ALL200_SUMMARY_CACHE_PATH = (
    ENSEMBLE_OUT_DIR
    / "focused_participant_semantic_summaries_ALL200_v1.json"
)

FN22_SUMMARY_CACHE_PATH = (
    ENSEMBLE_OUT_DIR
    / "focused_participant_semantic_summaries_FN22_v1.json"
)


# Load an existing partial ALL-200 cache, if one exists.
if FOCUSED_ALL200_SUMMARY_CACHE_PATH.exists():
    all200_focused_summary_cache = json.loads(
        FOCUSED_ALL200_SUMMARY_CACHE_PATH.read_text(
            encoding="utf-8"
        )
    )
else:
    all200_focused_summary_cache = {}


# Import the summaries already produced for the 22 diagnostic cases.
if FN22_SUMMARY_CACHE_PATH.exists():
    fn22_summary_cache = json.loads(
        FN22_SUMMARY_CACHE_PATH.read_text(
            encoding="utf-8"
        )
    )
else:
    fn22_summary_cache = {}


imported_summary_records = 0

for key, record in fn22_summary_cache.items():
    if (
        record.get("experiment_version")
        == FOCUSED_SUMMARY_VERSION
        and key not in all200_focused_summary_cache
    ):
        all200_focused_summary_cache[key] = copy.deepcopy(
            record
        )
        imported_summary_records += 1


# Replace the global focused cache used by the existing functions.
focused_summary_cache = all200_focused_summary_cache

# The existing save_focused_summary_cache() function reads this
# global path, so point it to the ALL-200 cache.
FOCUSED_SUMMARY_CACHE_PATH = (
    FOCUSED_ALL200_SUMMARY_CACHE_PATH
)


save_focused_summary_cache()


print("Total cases:", len(cases))
print(
    "Imported diagnostic summary records:",
    imported_summary_records,
)
print(
    "Existing ALL-200 summary records:",
    len(focused_summary_cache),
)
print(
    "ALL-200 summary cache:",
    FOCUSED_ALL200_SUMMARY_CACHE_PATH,
)

Total cases: 200
Imported diagnostic summary records: 0
Existing ALL-200 summary records: 400
ALL-200 summary cache: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/focused_participant_semantic_summaries_ALL200_v1.json


In [ ]:
# ============================================================
# BUILD UNIQUE FOCUSED-SUMMARY TASKS FOR ALL 200 CASES
# ============================================================

all200_focused_summary_tasks = {}


for case in cases:
    for participant_role in ["A", "B"]:

        participant_item = case[
            f"participant_{participant_role}"
        ]

        for segment_idx in [0, 1]:

            segment = resolve_participant_segment(
                participant_item=participant_item,
                segment_idx=segment_idx,
            )

            task_key = focused_summary_key(
                participant_item=participant_item,
                segment=segment,
            )

            all200_focused_summary_tasks[task_key] = {
                "participant_item": participant_item,
                "segment_idx": segment_idx,
            }


cached_task_count = sum(
    key in focused_summary_cache
    for key in all200_focused_summary_tasks
)

pending_task_count = (
    len(all200_focused_summary_tasks)
    - cached_task_count
)


print(
    "Unique participant-segment tasks:",
    len(all200_focused_summary_tasks),
)

print(
    "Already cached:",
    cached_task_count,
)

print(
    "Remaining video inferences:",
    pending_task_count,
)

Unique participant-segment tasks: 400
Already cached: 400
Remaining video inferences: 0


In [ ]:
# ============================================================
# GENERATE FOCUSED SEMANTIC SUMMARIES FOR ALL CASES
# ============================================================

for task_key, task in tqdm(
    all200_focused_summary_tasks.items(),
    desc="Focused semantic summaries — ALL 200",
):
    if task_key in focused_summary_cache:
        continue

    _ = analyze_focused_participant_segment(
        participant_item=task["participant_item"],
        segment_idx=task["segment_idx"],
        force_rerun=False,
    )


save_focused_summary_cache()


completed_task_count = sum(
    key in focused_summary_cache
    for key in all200_focused_summary_tasks
)


print(
    "Expected unique summaries:",
    len(all200_focused_summary_tasks),
)

print(
    "Available focused summaries:",
    completed_task_count,
)

print(
    "Saved:",
    FOCUSED_ALL200_SUMMARY_CACHE_PATH,
)

Expected unique summaries: 400
Available focused summaries: 400
Saved: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/focused_participant_semantic_summaries_ALL200_v1.json


In [ ]:
# ============================================================
# VALIDATE ALL FOCUSED PARTICIPANT SUMMARIES
# ============================================================

all200_focused_summary_errors = []


for task_key, task in all200_focused_summary_tasks.items():

    record = focused_summary_cache.get(
        task_key
    )

    participant_item = task[
        "participant_item"
    ]

    if record is None:
        all200_focused_summary_errors.append({
            "conversation_id": (
                participant_item["conversation_id"]
            ),
            "participant_id": (
                participant_item["participant_id"]
            ),
            "segment_idx": task["segment_idx"],
            "error": "missing focused summary",
        })
        continue

    parsed = record.get("parsed")

    if (
        not isinstance(parsed, dict)
        or parsed.get("parse_error")
    ):
        all200_focused_summary_errors.append({
            "conversation_id": (
                participant_item["conversation_id"]
            ),
            "participant_id": (
                participant_item["participant_id"]
            ),
            "segment_idx": task["segment_idx"],
            "error": "invalid parsed JSON",
            "raw_output": record.get("raw_output"),
        })
        continue

    required_semantic_fields = [
        "detailed_speech_summary",
        "main_topic",
        "secondary_topics",
        "key_semantic_details",
        "summary_specificity",
        "unclear_content",
    ]

    missing_fields = [
        field
        for field in required_semantic_fields
        if field not in parsed
    ]

    if missing_fields:
        all200_focused_summary_errors.append({
            "conversation_id": (
                participant_item["conversation_id"]
            ),
            "participant_id": (
                participant_item["participant_id"]
            ),
            "segment_idx": task["segment_idx"],
            "error": "missing semantic fields",
            "missing_fields": missing_fields,
        })


print(
    "Focused summary errors:",
    len(all200_focused_summary_errors),
)


if all200_focused_summary_errors:
    display(
        pd.DataFrame(
            all200_focused_summary_errors
        )
    )

    raise RuntimeError(
        "Some focused summaries are missing or invalid. "
        "Inspect the table before running classification."
    )

else:
    print(
        "All focused participant summaries are valid."
    )

Focused summary errors: 0
All focused participant summaries are valid.


## 9. Focused-Only 120-Second Classification

In [ ]:
# ============================================================
# FULL 200-CASE FOCUSED 120s CLASSIFIER CONFIGURATION
# ============================================================

FOCUSED_ALL200_CLASSIFIER_VERSION = (
    "focused_semantic_four_summaries_ALL200_final_v1"
)

FOCUSED_ALL200_RESULTS_PATH = (
    ENSEMBLE_OUT_DIR
    / "focused_semantic_120s_ALL200_final_v1.json"
)

FOCUSED_ALL200_CSV_PATH = (
    ENSEMBLE_OUT_DIR
    / "focused_semantic_120s_ALL200_final_v1.csv"
)

FOCUSED_ALL200_ERRORS_CSV_PATH = (
    ENSEMBLE_OUT_DIR
    / "focused_semantic_120s_ALL200_final_errors_v1.csv"
)

REUSE_FN22_CLASSIFICATIONS = False

# ------------------------------------------------------------
# Load partial/full ALL-200 results
# ------------------------------------------------------------

if FOCUSED_ALL200_RESULTS_PATH.exists():
    loaded_all200_results = json.loads(
        FOCUSED_ALL200_RESULTS_PATH.read_text(
            encoding="utf-8"
        )
    )
else:
    loaded_all200_results = []


focused_all200_results = [
    result
    for result in loaded_all200_results
    if (
        result.get("experiment_version")
        == FOCUSED_ALL200_CLASSIFIER_VERSION
    )
]


focused_all200_completed_lookup = {
    result["case_id"]: result
    for result in focused_all200_results
}


# ------------------------------------------------------------
# Reuse exact existing FN22 classifications
# ------------------------------------------------------------

reused_fn22_classifications = 0


if (
    REUSE_FN22_CLASSIFICATIONS
    and FN22_CLASSIFICATION_RESULTS_PATH.exists()
):
    fn22_classification_results = json.loads(
        FN22_CLASSIFICATION_RESULTS_PATH.read_text(
            encoding="utf-8"
        )
    )

    for old_result in fn22_classification_results:

        case_id = old_result.get("case_id")

        if (
            case_id is None
            or case_id in focused_all200_completed_lookup
        ):
            continue

        copied_result = copy.deepcopy(
            old_result
        )

        copied_result["experiment_version"] = (
            FOCUSED_ALL200_CLASSIFIER_VERSION
        )

        copied_result[
            "reused_from_fn22_diagnostic"
        ] = True

        focused_all200_results.append(
            copied_result
        )

        focused_all200_completed_lookup[
            case_id
        ] = copied_result

        reused_fn22_classifications += 1


FOCUSED_ALL200_RESULTS_PATH.write_text(
    json.dumps(
        focused_all200_results,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print(
    "Existing ALL-200 classifications:",
    len(focused_all200_completed_lookup),
)

print(
    "Imported FN22 classifications:",
    reused_fn22_classifications,
)

print(
    "Remaining classifications:",
    len(cases)
    - len(focused_all200_completed_lookup),
)

print(
    "Results path:",
    FOCUSED_ALL200_RESULTS_PATH,
)

Existing ALL-200 classifications: 200
Imported FN22 classifications: 0
Remaining classifications: 0
Results path: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/focused_semantic_120s_ALL200_final_v1.json


In [ ]:
# ============================================================
# RUN FOCUSED 120s CLASSIFICATION FOR ALL 200 CASES
# ============================================================

all200_cases_to_run = list(cases)

all200_rng = random.Random(42 + 9000)
all200_rng.shuffle(all200_cases_to_run)


for case in tqdm(
    all200_cases_to_run,
    desc="Focused 120s classification — ALL 200",
):
    case_id = case["case_id"]

    if case_id in focused_all200_completed_lookup:
        continue

    focused_input = build_focused_case_input(
        case
    )

    prompt = (
        FOCUSED_120S_COMPARISON_PROMPT_TEMPLATE
        .format(
            semantic_input=json.dumps(
                focused_input,
                indent=2,
                ensure_ascii=False,
            )
        )
    )

    raw_output = qwen_text_only(
        prompt=prompt,
        max_new_tokens=450,
    )

    parsed_output = extract_json_from_text(
        raw_output
    )

    result = {
        "experiment_version": (
            FOCUSED_ALL200_CLASSIFIER_VERSION
        ),

        "case_id": case_id,

        # Gold values are not included in the prompt.
        # They are saved only for later evaluation.
        "gold_label": case["label"],
        "gold_anomaly_type": (
            case["anomaly_type"]
        ),

        "participant_A_conversation": (
            case["participant_A"][
                "conversation_id"
            ]
        ),

        "participant_A_id": (
            case["participant_A"][
                "participant_id"
            ]
        ),

        "participant_B_conversation": (
            case["participant_B"][
                "conversation_id"
            ]
        ),

        "participant_B_id": (
            case["participant_B"][
                "participant_id"
            ]
        ),

        "focused_semantic_input": (
            focused_input
        ),

        "raw_output": raw_output,
        "parsed": parsed_output,

        "reused_from_fn22_diagnostic": False,
    }

    focused_all200_results.append(
        result
    )

    focused_all200_completed_lookup[
        case_id
    ] = result

    # Incremental save after every case.
    FOCUSED_ALL200_RESULTS_PATH.write_text(
        json.dumps(
            focused_all200_results,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


print(
    "Expected classifications:",
    len(cases),
)

print(
    "Completed classifications:",
    len(focused_all200_completed_lookup),
)

print(
    "Saved:",
    FOCUSED_ALL200_RESULTS_PATH,
)


assert len(
    focused_all200_completed_lookup
) == len(cases), (
    "Not all 200 cases were completed."
)

Expected classifications: 200
Completed classifications: 200
Saved: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/focused_semantic_120s_ALL200_final_v1.json


In [ ]:
# ============================================================
# EVALUATE FULL 200-CASE FOCUSED SEMANTIC EXPERIMENT
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)


case_lookup = {
    case["case_id"]: case
    for case in cases
}


focused_all200_rows = []


for case in cases:
    case_id = case["case_id"]

    result = focused_all200_completed_lookup.get(
        case_id
    )

    if result is None:
        focused_all200_rows.append({
            "case_id": case_id,
            "gold_label": case["label"],
            "gold_anomaly_type": (
                case["anomaly_type"]
            ),
            "predicted_label": None,
            "parse_error": True,
        })
        continue

    parsed = result.get("parsed", {})

    if not isinstance(parsed, dict):
        parsed = {}

    focused_all200_rows.append({
        "case_id": case_id,

        "gold_label": case["label"],
        "gold_anomaly_type": (
            case["anomaly_type"]
        ),

        "predicted_label": (
            normalize_focused_prediction(
                parsed
            )
        ),

        "predicted_anomaly_type": (
            parsed.get("anomaly_type")
        ),

        "confidence": (
            parsed.get("confidence")
        ),

        "segment_0_assessment": (
            parsed.get(
                "segment_0_assessment"
            )
        ),

        "segment_1_assessment": (
            parsed.get(
                "segment_1_assessment"
            )
        ),

        "cross_segment_assessment": (
            parsed.get(
                "cross_segment_assessment"
            )
        ),

        "decisive_evidence": (
            parsed.get(
                "decisive_evidence"
            )
        ),

        "reasoning": (
            parsed.get("reasoning")
        ),

        "parse_error": (
            parsed.get(
                "parse_error",
                False,
            )
        ),

        "reused_from_fn22_diagnostic": (
            result.get(
                "reused_from_fn22_diagnostic",
                False,
            )
        ),
    })


focused_all200_df = (
    pd.DataFrame(
        focused_all200_rows
    )
    .sort_values("case_id")
    .reset_index(drop=True)
)


valid_focused_all200 = (
    focused_all200_df[
        focused_all200_df[
            "predicted_label"
        ].isin(
            ["NORMAL", "ANOMALOUS"]
        )
    ]
    .copy()
)


print(
    "Total expected cases:",
    len(cases),
)

print(
    "Total result rows:",
    len(focused_all200_df),
)

print(
    "Valid parsed predictions:",
    len(valid_focused_all200),
)

print(
    "Invalid predictions:",
    len(focused_all200_df)
    - len(valid_focused_all200),
)


assert len(focused_all200_df) == 200

assert len(valid_focused_all200) == 200, (
    "Some outputs could not be parsed or normalized."
)


focused_all200_accuracy = accuracy_score(
    valid_focused_all200["gold_label"],
    valid_focused_all200["predicted_label"],
)


print(
    "\nAccuracy:",
    focused_all200_accuracy,
)


print("\nClassification report:")

print(
    classification_report(
        valid_focused_all200["gold_label"],
        valid_focused_all200[
            "predicted_label"
        ],
        labels=[
            "ANOMALOUS",
            "NORMAL",
        ],
        zero_division=0,
    )
)


focused_all200_cm = confusion_matrix(
    valid_focused_all200["gold_label"],
    valid_focused_all200[
        "predicted_label"
    ],
    labels=[
        "NORMAL",
        "ANOMALOUS",
    ],
)


print(
    "\nConfusion matrix "
    "[rows=Gold, columns=Predicted]"
)

print(
    "Order: [NORMAL, ANOMALOUS]"
)

print(focused_all200_cm)


# ------------------------------------------------------------
# Error sets
# ------------------------------------------------------------

focused_all200_false_negatives = (
    valid_focused_all200[
        (
            valid_focused_all200[
                "gold_label"
            ] == "ANOMALOUS"
        )
        & (
            valid_focused_all200[
                "predicted_label"
            ] == "NORMAL"
        )
    ]
    .copy()
    .reset_index(drop=True)
)


focused_all200_false_positives = (
    valid_focused_all200[
        (
            valid_focused_all200[
                "gold_label"
            ] == "NORMAL"
        )
        & (
            valid_focused_all200[
                "predicted_label"
            ] == "ANOMALOUS"
        )
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "\nMissed wrong partners:",
    len(focused_all200_false_negatives),
)

print(
    "Normal cases predicted anomalous:",
    len(focused_all200_false_positives),
)


# ------------------------------------------------------------
# Save evaluation tables
# ------------------------------------------------------------

focused_all200_df.to_csv(
    FOCUSED_ALL200_CSV_PATH,
    index=False,
)


focused_all200_errors_df = (
    pd.concat(
        [
            focused_all200_false_negatives.assign(
                error_type="false_negative"
            ),
            focused_all200_false_positives.assign(
                error_type="false_positive"
            ),
        ],
        ignore_index=True,
    )
)


focused_all200_errors_df.to_csv(
    FOCUSED_ALL200_ERRORS_CSV_PATH,
    index=False,
)


print(
    "\nSaved full evaluation:",
    FOCUSED_ALL200_CSV_PATH,
)

print(
    "Saved errors:",
    FOCUSED_ALL200_ERRORS_CSV_PATH,
)

Total expected cases: 200
Total result rows: 200
Valid parsed predictions: 200
Invalid predictions: 0

Accuracy: 0.735

Classification report:
              precision    recall  f1-score   support

   ANOMALOUS       0.65      1.00      0.79       100
      NORMAL       1.00      0.47      0.64       100

    accuracy                           0.73       200
   macro avg       0.83      0.73      0.71       200
weighted avg       0.83      0.73      0.71       200


Confusion matrix [rows=Gold, columns=Predicted]
Order: [NORMAL, ANOMALOUS]
[[ 47  53]
 [  0 100]]

Missed wrong partners: 0
Normal cases predicted anomalous: 53

Saved full evaluation: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/focused_semantic_120s_ALL200_final_v1.csv
Saved errors: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/focused_semantic_120s_ALL200_final_errors_v1.csv


### Focused-only result

The focused representation detects **100/100 Wrong Partner** cases but preserves only **47/100 NORMAL** conversations, producing an overall accuracy of **73.5%**.

Greater semantic specificity therefore substantially increases mismatch sensitivity, but it can also over-interpret legitimate differences in detail, perspective, or conversational contribution as evidence that the participants do not belong together.

# Experiment 3 — Coarse + Focused Semantic Evidence

The final reported isolated semantic experiment combines both resolutions.

Each participant segment contains:

**Coarse evidence**
- `speech_content_summary`
- `apparent_topic`

**Focused evidence**
- `speaks`
- `detailed_speech_summary`
- `main_topic`
- `secondary_topics`
- `key_semantic_details`
- `summary_specificity`
- `unclear_content`
- `confidence`

The original coarse semantic decision prompt is retained, with its allowed input fields expanded to include the focused representation. No previous prediction or gold label is supplied to the model.

In [ ]:
# ============================================================
# NEW EXPERIMENT:
# COARSE SUMMARIES + FULL FOCUSED SUMMARIES
# WITH THE ORIGINAL COARSE DECISION PROMPT
# ============================================================

from pathlib import Path
from tqdm.auto import tqdm

import copy
import json
import random
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)


required_combined_objects = [
    "cases",
    "ENSEMBLE_OUT_DIR",
    "SEMANTIC_120S_PROMPT_TEMPLATE",
    "compatible_existing_results",
    "semantic_120s_df",
    "normalize_semantic_120s_prediction",
    "resolve_participant_segment",
    "focused_summary_key",
    "qwen_text_only",
    "extract_json_from_text",
]

missing_combined_objects = [
    name
    for name in required_combined_objects
    if name not in globals()
]

if missing_combined_objects:
    raise RuntimeError(
        "Missing required notebook objects: "
        f"{missing_combined_objects}"
    )


assert len(cases) == 200
assert len(compatible_existing_results) == 200


# ------------------------------------------------------------
# Verify the exact coarse baseline
# ------------------------------------------------------------

coarse_valid_for_combined = (
    semantic_120s_df[
        semantic_120s_df[
            "predicted_label"
        ].isin(
            ["NORMAL", "ANOMALOUS"]
        )
    ]
    .copy()
)


coarse_baseline_cm_for_combined = confusion_matrix(
    coarse_valid_for_combined["gold_label"],
    coarse_valid_for_combined["predicted_label"],
    labels=["NORMAL", "ANOMALOUS"],
)


print(
    "Stored coarse confusion matrix "
    "[rows=Gold, columns=Predicted]:"
)

print(coarse_baseline_cm_for_combined)


expected_coarse_cm = np.array([
    [91, 9],
    [22, 78],
])


assert np.array_equal(
    coarse_baseline_cm_for_combined,
    expected_coarse_cm,
), (
    "The loaded coarse experiment does not match "
    "the expected [[91, 9], [22, 78]] result."
)

Stored coarse confusion matrix [rows=Gold, columns=Predicted]:
[[91  9]
 [22 78]]


In [ ]:
# ============================================================
# LOAD EXISTING ALL-200 FOCUSED SUMMARY CACHE
# NO VIDEO INFERENCE
# ============================================================

if "FOCUSED_ALL200_SUMMARY_CACHE_PATH" in globals():

    COMBINED_FOCUSED_CACHE_PATH = Path(
        FOCUSED_ALL200_SUMMARY_CACHE_PATH
    )

else:

    COMBINED_FOCUSED_CACHE_PATH = (
        ENSEMBLE_OUT_DIR
        / "focused_participant_semantic_summaries_ALL200_v1.json"
    )


if not COMBINED_FOCUSED_CACHE_PATH.exists():
    raise FileNotFoundError(
        "Focused summary cache not found:\n"
        f"{COMBINED_FOCUSED_CACHE_PATH}"
    )


combined_focused_cache = json.loads(
    COMBINED_FOCUSED_CACHE_PATH.read_text(
        encoding="utf-8"
    )
)


print(
    "Loaded focused cache records:",
    len(combined_focused_cache),
)

print(
    "Focused cache:",
    COMBINED_FOCUSED_CACHE_PATH,
)

Loaded focused cache records: 400
Focused cache: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/focused_participant_semantic_summaries_ALL200_v1.json


In [ ]:
# ============================================================
# FULL FOCUSED SEMANTIC PROJECTION
# ============================================================

FULL_FOCUSED_SEMANTIC_FIELDS = [
    "speaks",
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


def safe_full_focused_projection(record):
    """
    Keeps all useful focused semantic fields.

    Invalid summaries become explicit unclear evidence.
    """

    parsed = (
        record.get("parsed")
        if isinstance(record, dict)
        else None
    )

    if (
        not isinstance(parsed, dict)
        or parsed.get("parse_error", False)
    ):
        return {
            "speaks": None,
            "detailed_speech_summary": "unclear",
            "main_topic": "unclear",
            "secondary_topics": [],
            "key_semantic_details": [],
            "summary_specificity": "low",
            "unclear_content": (
                "The focused summary could not "
                "be parsed reliably."
            ),
            "confidence": 0.0,
        }


    projection = {
        field: parsed.get(field)
        for field in FULL_FOCUSED_SEMANTIC_FIELDS
    }


    # Normalize text fields.
    for field in [
        "detailed_speech_summary",
        "main_topic",
        "summary_specificity",
        "unclear_content",
    ]:
        value = projection.get(field)

        if value is None or not str(value).strip():
            projection[field] = "unclear"

        else:
            projection[field] = str(value).strip()


    # Normalize list fields.
    for field in [
        "secondary_topics",
        "key_semantic_details",
    ]:
        value = projection.get(field)

        if not isinstance(value, list):
            projection[field] = []


    # Normalize confidence.
    try:
        projection["confidence"] = float(
            projection.get("confidence", 0.0)
        )

    except Exception:
        projection["confidence"] = 0.0


    return projection

In [ ]:
# ============================================================
# GET EXISTING FULL FOCUSED SUMMARY
# ============================================================

def get_existing_full_focused_summary(
    participant_item,
    segment_idx,
):
    """
    Retrieves a focused summary only from the saved cache.

    This function never invokes Qwen video inference.
    """

    segment = resolve_participant_segment(
        participant_item=participant_item,
        segment_idx=segment_idx,
    )

    cache_key = focused_summary_key(
        participant_item=participant_item,
        segment=segment,
    )


    record = combined_focused_cache.get(
        cache_key
    )


    # Metadata fallback in case cache-key versions differ.
    if record is None:

        conversation_id = (
            participant_item["conversation_id"]
        )

        participant_id = (
            participant_item["participant_id"]
        )

        matches = [
            candidate
            for candidate in combined_focused_cache.values()
            if (
                isinstance(candidate, dict)
                and candidate.get("conversation_id")
                == conversation_id
                and candidate.get("participant_id")
                == participant_id
                and int(
                    candidate.get("segment_idx", -1)
                ) == int(segment_idx)
            )
        ]

        if not matches:
            raise KeyError(
                "No focused summary found for:\n"
                f"conversation_id={conversation_id}\n"
                f"participant_id={participant_id}\n"
                f"segment_idx={segment_idx}"
            )

        record = matches[-1]


    return safe_full_focused_projection(
        record
    )

In [ ]:
# ============================================================
# BUILD COARSE + FULL-FOCUSED INPUT FOR ONE CASE
# ============================================================

SEGMENT_KEYS = {
    0: "segment_0_0_to_60_seconds",
    1: "segment_1_60_to_120_seconds",
}


def build_coarse_plus_full_focused_input(case):
    """
    Uses:
      - existing coarse semantic input
      - existing full focused semantic input

    No previous prediction and no gold label are included.
    """

    case_id = case["case_id"]

    if case_id not in compatible_existing_results:
        raise KeyError(
            f"Missing saved coarse result for {case_id}."
        )


    coarse_result = compatible_existing_results[
        case_id
    ]

    coarse_input = coarse_result.get(
        "semantic_input"
    )


    if not isinstance(coarse_input, dict):
        raise RuntimeError(
            f"Invalid coarse semantic input for {case_id}."
        )


    combined_input = {
        "participant_A": {},
        "participant_B": {},
    }


    for participant_role in ["A", "B"]:

        participant_key = (
            f"participant_{participant_role}"
        )

        participant_item = case[
            participant_key
        ]

        coarse_participant = coarse_input.get(
            participant_key,
            {},
        )


        for segment_idx, segment_key in SEGMENT_KEYS.items():

            coarse_segment = coarse_participant.get(
                segment_key,
                {},
            )


            focused_segment = (
                get_existing_full_focused_summary(
                    participant_item=participant_item,
                    segment_idx=segment_idx,
                )
            )


            combined_input[
                participant_key
            ][segment_key] = {
                # Original coarse information.
                "speech_content_summary": (
                    coarse_segment.get(
                        "speech_content_summary",
                        "unclear",
                    )
                ),

                "apparent_topic": (
                    coarse_segment.get(
                        "apparent_topic",
                        "unclear",
                    )
                ),

                # All useful fields from the focused summary.
                "focused_summary": (
                    focused_segment
                ),
            }


    return combined_input

In [ ]:
# ============================================================
# VALIDATE ALL 200 COMBINED INPUTS
# ============================================================

combined_input_errors = []


for case in cases:

    try:
        combined_input = (
            build_coarse_plus_full_focused_input(
                case
            )
        )

        # Ensure it can be serialized for the prompt.
        _ = json.dumps(
            combined_input,
            ensure_ascii=False,
        )

    except Exception as exc:
        combined_input_errors.append({
            "case_id": case["case_id"],
            "error": str(exc),
        })


print(
    "Combined-input errors:",
    len(combined_input_errors),
)


if combined_input_errors:

    display(
        pd.DataFrame(
            combined_input_errors
        )
    )

    raise RuntimeError(
        "Some cases do not have valid "
        "coarse + focused inputs."
    )

else:
    print(
        "All 200 cases have valid "
        "coarse + full-focused inputs."
    )

Combined-input errors: 0
All 200 cases have valid coarse + full-focused inputs.


In [ ]:
# ============================================================
# ORIGINAL COARSE PROMPT LOGIC
# WITH COARSE + FULL FOCUSED INPUT FIELDS
# ============================================================

ORIGINAL_ALLOWED_FIELDS_BLOCK = """You must use ONLY:
- speech_content_summary
- apparent_topic
"""


NEW_ALLOWED_FIELDS_BLOCK = """You must use ONLY the supplied semantic fields.

For each participant segment, the input contains:

COARSE SUMMARY:
- speech_content_summary
- apparent_topic

FOCUSED SUMMARY:
- speaks
- detailed_speech_summary
- main_topic
- secondary_topics
- key_semantic_details
- summary_specificity
- unclear_content
- confidence

The coarse and focused fields are two independently generated semantic
descriptions of the SAME participant segment. Interpret them together.
Do not treat differences between a participant's own coarse and focused
summaries as evidence of wrong partner.
"""


assert (
    ORIGINAL_ALLOWED_FIELDS_BLOCK
    in SEMANTIC_120S_PROMPT_TEMPLATE
), (
    "The original field block was not found inside "
    "SEMANTIC_120S_PROMPT_TEMPLATE."
)


COARSE_PLUS_FULL_FOCUSED_PROMPT_V1 = (
    SEMANTIC_120S_PROMPT_TEMPLATE.replace(
        ORIGINAL_ALLOWED_FIELDS_BLOCK,
        NEW_ALLOWED_FIELDS_BLOCK,
        1,
    )
)


print(
    COARSE_PLUS_FULL_FOCUSED_PROMPT_V1[
        :1800
    ]
)

In [ ]:
# ============================================================
# COARSE + FULL FOCUSED / ORIGINAL PROMPT EXPERIMENT
# ============================================================

COARSE_FULL_FOCUSED_EXPERIMENT_VERSION = (
    "coarse_plus_full_focused_original_prompt_ALL200_v1"
)

COARSE_FULL_FOCUSED_RESULTS_PATH = (
    ENSEMBLE_OUT_DIR
    / "coarse_plus_full_focused_original_prompt_ALL200_v1.json"
)

COARSE_FULL_FOCUSED_CSV_PATH = (
    ENSEMBLE_OUT_DIR
    / "coarse_plus_full_focused_original_prompt_ALL200_v1.csv"
)

COARSE_FULL_FOCUSED_ERRORS_PATH = (
    ENSEMBLE_OUT_DIR
    / "coarse_plus_full_focused_original_prompt_ALL200_v1_errors.csv"
)


if COARSE_FULL_FOCUSED_RESULTS_PATH.exists():

    coarse_full_focused_results = json.loads(
        COARSE_FULL_FOCUSED_RESULTS_PATH.read_text(
            encoding="utf-8"
        )
    )

else:
    coarse_full_focused_results = []


coarse_full_focused_results = [
    result
    for result in coarse_full_focused_results
    if (
        result.get("experiment_version")
        == COARSE_FULL_FOCUSED_EXPERIMENT_VERSION
    )
]


coarse_full_focused_completed_lookup = {
    result["case_id"]: result
    for result in coarse_full_focused_results
}


print(
    "Existing experiment results:",
    len(coarse_full_focused_completed_lookup),
)

print(
    "Remaining classifications:",
    len(cases)
    - len(coarse_full_focused_completed_lookup),
)

print(
    "Results path:",
    COARSE_FULL_FOCUSED_RESULTS_PATH,
)

Existing experiment results: 0
Remaining classifications: 200
Results path: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/coarse_plus_full_focused_original_prompt_ALL200_v1.json


In [ ]:
# ============================================================
# RUN COARSE + FULL-FOCUSED CLASSIFICATION
# USING THE ORIGINAL PROMPT LOGIC
# ============================================================

combined_cases_to_run = list(cases)

combined_rng = random.Random(
    42 + 70000
)

combined_rng.shuffle(
    combined_cases_to_run
)


for case in tqdm(
    combined_cases_to_run,
    desc="Coarse + full focused / original prompt",
):

    case_id = case["case_id"]


    if case_id in coarse_full_focused_completed_lookup:
        continue


    combined_semantic_input = (
        build_coarse_plus_full_focused_input(
            case
        )
    )


    prompt = (
        COARSE_PLUS_FULL_FOCUSED_PROMPT_V1
        .format(
            semantic_input=json.dumps(
                combined_semantic_input,
                indent=2,
                ensure_ascii=False,
            )
        )
    )


    # Same output length as the original coarse experiment.
    raw_output = qwen_text_only(
        prompt=prompt,
        max_new_tokens=350,
    )


    parsed_output = extract_json_from_text(
        raw_output
    )


    result = {
        "experiment_version": (
            COARSE_FULL_FOCUSED_EXPERIMENT_VERSION
        ),

        "case_id": case_id,

        # Stored after inference only.
        # Never included in the prompt.
        "gold_label": case["label"],

        "gold_anomaly_type": (
            case["anomaly_type"]
        ),

        "combined_semantic_input": (
            combined_semantic_input
        ),

        "raw_output": raw_output,

        "parsed": parsed_output,
    }


    coarse_full_focused_results.append(
        result
    )


    coarse_full_focused_completed_lookup[
        case_id
    ] = result


    # Incremental save.
    COARSE_FULL_FOCUSED_RESULTS_PATH.write_text(
        json.dumps(
            coarse_full_focused_results,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


print(
    "Expected classifications:",
    len(cases),
)

print(
    "Completed classifications:",
    len(coarse_full_focused_completed_lookup),
)

print(
    "Saved:",
    COARSE_FULL_FOCUSED_RESULTS_PATH,
)


assert len(
    coarse_full_focused_completed_lookup
) == 200

Expected classifications: 200
Completed classifications: 200
Saved: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/coarse_plus_full_focused_original_prompt_ALL200_v1.json


In [ ]:
# ============================================================
# EVALUATE COARSE + FULL FOCUSED / ORIGINAL PROMPT
# ============================================================

coarse_full_focused_rows = []


for case in cases:

    case_id = case["case_id"]

    result = (
        coarse_full_focused_completed_lookup.get(
            case_id
        )
    )


    if result is None:

        coarse_full_focused_rows.append({
            "case_id": case_id,
            "gold_label": case["label"],
            "predicted_label": None,
            "parse_error": True,
        })

        continue


    parsed = result.get(
        "parsed",
        {},
    )


    if not isinstance(parsed, dict):
        parsed = {}


    coarse_full_focused_rows.append({
        "case_id": case_id,

        "gold_label": case["label"],

        "gold_anomaly_type": (
            case["anomaly_type"]
        ),

        "predicted_label": (
            normalize_semantic_120s_prediction(
                parsed
            )
        ),

        "predicted_anomaly_type": (
            parsed.get("anomaly_type")
        ),

        "confidence": (
            parsed.get("confidence")
        ),

        "segment_0_assessment": (
            parsed.get(
                "segment_0_assessment"
            )
        ),

        "segment_1_assessment": (
            parsed.get(
                "segment_1_assessment"
            )
        ),

        "cross_segment_assessment": (
            parsed.get(
                "cross_segment_assessment"
            )
        ),

        "reasoning": (
            parsed.get("reasoning")
        ),

        "parse_error": (
            parsed.get(
                "parse_error",
                False,
            )
        ),
    })


coarse_full_focused_df = (
    pd.DataFrame(
        coarse_full_focused_rows
    )
    .sort_values("case_id")
    .reset_index(drop=True)
)


valid_coarse_full_focused = (
    coarse_full_focused_df[
        coarse_full_focused_df[
            "predicted_label"
        ].isin(
            ["NORMAL", "ANOMALOUS"]
        )
    ]
    .copy()
)


print(
    "Total rows:",
    len(coarse_full_focused_df),
)

print(
    "Valid predictions:",
    len(valid_coarse_full_focused),
)

print(
    "Invalid predictions:",
    len(coarse_full_focused_df)
    - len(valid_coarse_full_focused),
)


assert len(coarse_full_focused_df) == 200

assert len(valid_coarse_full_focused) == 200, (
    "Some outputs could not be normalized."
)


coarse_full_focused_accuracy = accuracy_score(
    valid_coarse_full_focused[
        "gold_label"
    ],

    valid_coarse_full_focused[
        "predicted_label"
    ],
)


coarse_full_focused_cm = confusion_matrix(
    valid_coarse_full_focused[
        "gold_label"
    ],

    valid_coarse_full_focused[
        "predicted_label"
    ],

    labels=[
        "NORMAL",
        "ANOMALOUS",
    ],
)


print(
    "\nAccuracy:",
    coarse_full_focused_accuracy,
)


print(
    "\nClassification report:"
)

print(
    classification_report(
        valid_coarse_full_focused[
            "gold_label"
        ],

        valid_coarse_full_focused[
            "predicted_label"
        ],

        labels=[
            "ANOMALOUS",
            "NORMAL",
        ],

        zero_division=0,
    )
)


print(
    "\nConfusion matrix "
    "[rows=Gold, columns=Predicted]"
)

print(
    "Order: [NORMAL, ANOMALOUS]"
)

print(
    coarse_full_focused_cm
)

Total rows: 200
Valid predictions: 200
Invalid predictions: 0

Accuracy: 0.865

Classification report:
              precision    recall  f1-score   support

   ANOMALOUS       0.81      0.95      0.88       100
      NORMAL       0.94      0.78      0.85       100

    accuracy                           0.86       200
   macro avg       0.88      0.86      0.86       200
weighted avg       0.88      0.86      0.86       200


Confusion matrix [rows=Gold, columns=Predicted]
Order: [NORMAL, ANOMALOUS]
[[78 22]
 [ 5 95]]


### Coarse + focused result

The combined representation correctly classifies **78/100 NORMAL** and **95/100 Wrong Partner** interactions, achieving the highest overall accuracy of the three semantic configurations: **86.5%**.

The two representations exhibit complementary behaviour. Coarse summaries provide broader conversational context and stronger NORMAL preservation, while focused summaries expose concrete semantic mismatches more aggressively. Combining them produces the strongest balance between the two classes.

# Final Isolated Semantic Comparison

| Representation | NORMAL | WRONG | Accuracy |
|---|---:|---:|---:|
| **Coarse only** | **91/100** | **78/100** | **84.5%** |
| **Focused only** | **47/100** | **100/100** | **73.5%** |
| **Coarse + focused** | **78/100** | **95/100** | **86.5%** |

The isolated semantic branch therefore demonstrates that participant-centric summaries contain strong information for detecting a genuinely relational anomaly: neither participant is individually abnormal, but their semantic content is inconsistent as a dyadic interaction.

The **coarse + focused** representation provides the best overall balance and is the isolated semantic configuration carried forward as evidence that useful Wrong Partner information exists before unified multi-branch consolidation.